In [ ]:
%pip install boto3 "sagemaker<3.0" pyathena pandas

In [ ]:
# import os

import os
os._exit(00)

In [1]:
import os
import boto3
import sagemaker

# Suppress v2 deprecation warning if on SageMaker v2
os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


In [2]:
# Initialize low-level AWS session (resolves credentials & region from IAM role)
session = boto3.session.Session()

# Extract current active AWS region (e.g., 'us-east-1')
region = session.region_name

# Initialize high-level SageMaker context (wraps boto3 clients)
sagemaker_session = sagemaker.Session()

# create 'sagemaker-{region}-{account_id}'
bucket = sagemaker_session.default_bucket()

In [3]:
# Define S3 prefix and paths
s3_data_prefix = "homework-2-1/data"
s3_staging_dir = f"s3://{bucket}/athena/staging/"

In [4]:
# Copy homework dataset to S3 bucket
!aws s3 cp ../../aai-540-homework/homework-2-1/data/ s3://{bucket}/{s3_data_prefix}/ --recursive

upload: ../../aai-540-homework/homework-2-1/data/dataset.csv to s3://sagemaker-us-east-1-682123396235/homework-2-1/data/dataset.csv
upload: ../../aai-540-homework/homework-2-1/data/.ipynb_checkpoints/dataset-checkpoint.csv to s3://sagemaker-us-east-1-682123396235/homework-2-1/data/.ipynb_checkpoints/dataset-checkpoint.csv


In [5]:
import warnings
warnings.filterwarnings(
    "ignore",
    message=".*pandas only supports SQLAlchemy connectable.*",
    category=UserWarning
)

In [6]:
import warnings
from pyathena import connect
import pandas as pd

# Suppress pandas DBAPI connection warning
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

database_name = "dsoaws_hw"
table_name_csv = "spotify_tracks"

# Connect to Athena
conn = connect(
    s3_staging_dir=s3_staging_dir,
    region_name=region
)

# Create Database
create_db_statement = f"CREATE DATABASE IF NOT EXISTS {database_name}"
pd.read_sql(create_db_statement, conn)

""


In [7]:
# Verify database creation
databases = pd.read_sql("SHOW DATABASES", conn)

if database_name in databases.iloc[:, 0].values:
    print(f"Database '{database_name}' successfully created/verified!")
else:
    print(f"Warning: Database '{database_name}' was not found in SHOW DATABASES.")

Database 'dsoaws_hw' successfully created/verified!


In [8]:
statement = "SHOW DATABASES"

df_show = pd.read_sql(statement, conn)
df_show.head(5)

,database_name
0,default
1,dsoaws
2,dsoaws_hw


In [9]:
# Drop existing table so schema changes take effect (IF NOT EXISTS would keep the old broken schema)
pd.read_sql(f"DROP TABLE IF EXISTS {database_name}.{table_name_csv}", conn)

# CSV has a leading unnamed index column (from pandas to_csv with index=True),
# so we include row_id as the first column to keep the rest aligned.
create_table_statement = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.{table_name_csv} (
    row_id INT,
    track_id STRING,
    artists STRING,
    album_name STRING,
    track_name STRING,
    popularity INT,
    duration_ms INT,
    explicit BOOLEAN,
    danceability DOUBLE,
    energy DOUBLE,
    key INT,
    loudness DOUBLE,
    mode INT,
    speechiness DOUBLE,
    acousticness DOUBLE,
    instrumentalness DOUBLE,
    liveness DOUBLE,
    valence DOUBLE,
    tempo DOUBLE,
    time_signature INT,
    track_genre STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LINES TERMINATED BY '\\n'
LOCATION 's3://{bucket}/{s3_data_prefix}/'
TBLPROPERTIES ('skip.header.line.count'='1')
"""

pd.read_sql(create_table_statement, conn)

# Verify table creation
df_show = pd.read_sql(f"SHOW TABLES IN {database_name}", conn)
print(df_show)

         tab_name
0  spotify_tracks


In [10]:
base_query = f"SELECT * FROM {database_name}.{table_name_csv}"
df_all = pd.read_sql(base_query, conn)

# Preview the first 5 rows
print(df_all.head())

# Check the row and column count
print(f"Loaded {df_all.shape[0]:,} rows and {df_all.shape[1]} columns.")

# Sanity check: columns should be aligned (artists = names, genre = labels, popularity numeric)
print("\nSanity check (expect artist names, acoustic-like genres, popularity ~0-100):")
print(df_all[['artists', 'track_name', 'popularity', 'track_genre']].head())

   row_id                track_id                 artists  \
0       0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1       1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2       2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3       3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4       4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms explicit  danceability  \
0                      Comedy        73.0     230666.0    False         0.676   
1            Ghost - Acoustic        55.0     149610.0    False         0.420   
2              To Begin Again        57.0     210826.

In [11]:
statement_1 = f"""
SELECT artists, track_name, popularity
FROM {database_name}.{table_name_csv}
WHERE popularity >= 99
"""
print("SQL Answer", "\n")
df_sql_1 = pd.read_sql(statement_1, conn)
print(df_sql_1)

SQL Answer 

                artists                 track_name  popularity
0  Sam Smith;Kim Petras  Unholy (feat. Kim Petras)         100
1     Charlie Brown Jr.               Prazo Longo"         333
2          Smyang Piano                    Vol. 4"      134340
3  Sam Smith;Kim Petras  Unholy (feat. Kim Petras)         100


In [12]:
df_pds_1 = df_all[df_all['popularity'] >= 99][['artists', 'track_name', 'popularity']]

print("Pandas Answer", "\n")
print(df_pds_1)

Pandas Answer 

                    artists                 track_name  popularity
20001  Sam Smith;Kim Petras  Unholy (feat. Kim Petras)       100.0
47884     Charlie Brown Jr.               Prazo Longo"       333.0
79571          Smyang Piano                    Vol. 4"    134340.0
81051  Sam Smith;Kim Petras  Unholy (feat. Kim Petras)       100.0


In [13]:
statement_2 = f"""
SELECT artists, AVG(popularity) AS avg_popularity
FROM {database_name}.{table_name_csv}
GROUP BY artists
HAVING AVG(popularity) = 92
"""
print("SQL Answer", "\n")
df_sql_2 = pd.read_sql(statement_2, conn)
print(df_sql_2)

SQL Answer 

             artists  avg_popularity
0       Harry Styles            92.0
1  Rema;Selena Gomez            92.0


In [14]:
df_pds_2 = (
    df_all.groupby('artists')['popularity']
    .mean()
    .reset_index()
)

print("Pandas Answer", "\n")
df_pds_2 = df_pds_2[df_pds_2['popularity'] == 92]
print(df_pds_2)

Pandas Answer 

                 artists  popularity
11567       Harry Styles        92.0
22880  Rema;Selena Gomez        92.0


In [15]:
statement_3 = f"""
SELECT track_genre, AVG(energy) AS avg_energy
FROM {database_name}.{table_name_csv}
GROUP BY track_genre
ORDER BY avg_energy DESC
LIMIT 10
"""

print("SQL Answer", "\n")
df_sql_3 = pd.read_sql(statement_3, conn)
print(df_sql_3)

SQL Answer 

  track_genre  avg_energy
0       0.797   1174026.0
1       0.556    691306.0
2      0.0371    629420.0
3      0.0359    614791.0
4       0.492    542000.0
5        0.45    538160.0
6       0.914    531293.0
7      0.0427    526946.0
8      0.0761    502786.0
9      0.0346    500088.0


In [16]:
df_pds_3 = (
    df_all.groupby('track_genre')['energy']
    .mean()
    .reset_index()
    .sort_values(by='energy', ascending=False)
    .head(10)
)

print("Pandas Answer", "\n")
print(df_pds_3)

Pandas Answer 

    track_genre     energy
247       0.797  1174026.0
209       0.556   691306.0
26       0.0371   629420.0
24       0.0359   614791.0
199       0.492   542000.0
192        0.45   538160.0
278       0.914   531293.0
39       0.0427   526946.0
68       0.0761   502786.0
18       0.0346   500088.0


In [17]:
statement_4 = f"""
SELECT COUNT(*) AS bad_bunny_track_count
FROM {database_name}.{table_name_csv}
WHERE artists LIKE '%Bad Bunny%'
"""

print("SQL Answer", "\n")
df_sql_4 = pd.read_sql(statement_4, conn)
print(df_sql_4)

SQL Answer 

   bad_bunny_track_count
0                    416


In [18]:
count_pds_4 = df_all['artists'].str.contains('Bad Bunny', case=False, na=False).sum()

print("Pandas Answer", "\n")
print(f"Bad Bunny track count: {count_pds_4}")

Pandas Answer 

Bad Bunny track count: 416


In [19]:
statement_5 = f"""
SELECT track_genre, MAX(popularity) AS max_track_popularity, AVG(popularity) AS avg_genre_popularity
FROM {database_name}.{table_name_csv}
GROUP BY track_genre
ORDER BY max_track_popularity DESC
LIMIT 10
"""

print("SQL Answer", "\n")
df_sql_5 = pd.read_sql(statement_5, conn)
print(df_sql_5)

SQL Answer 

  track_genre  max_track_popularity  avg_genre_popularity
0           4                134340          33673.750000
1       dance                   100             22.052201
2         pop                   100             47.477204
3       latin                    98              8.107472
4         edm                    98             34.758479
5   reggaeton                    98             23.825203
6      reggae                    98             20.590264
7      latino                    98             25.486789
8        rock                    96             18.851813
9       piano                    96             45.970033


In [20]:
df_pds_5 = (
    df_all.groupby('track_genre')['popularity']
    .agg(max_track_popularity='max', avg_genre_popularity='mean')
    .reset_index()
    .sort_values(by='max_track_popularity', ascending=False)
    .head(10)
)

print("Pandas Answer", "\n")
print(df_pds_5)

Pandas Answer 

     track_genre  max_track_popularity  avg_genre_popularity
901            4              134340.0          33673.750000
1280         pop                 100.0             47.477204
1220       dance                 100.0             22.052201
1268      latino                  98.0             25.486789
1267       latin                  98.0              8.107472
1289   reggaeton                  98.0             23.825203
1230         edm                  98.0             34.758479
1288      reggae                  98.0             20.590264
1279       piano                  96.0             45.970033
1290        rock                  96.0             18.851813


In [ ]:
# Release Resources

In [1]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

In [ ]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}